# IMC Prosperity Round 3 — ML Fair-Value Pipeline

Train a per-product fair-value model from 3 days of orderbook data and emit
lightweight coefficients usable from `trader_round3.py` (no sklearn at submit time).

Run top-to-bottom in Colab. Train = day 0+1, test = day 2.

## 1. Setup

In [ ]:
import os, json, pickle, math, warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

warnings.filterwarnings('ignore')

# Try lightgbm; fall back to GradientBoostingRegressor
try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    from sklearn.ensemble import GradientBoostingRegressor
    HAS_LGB = False
print('lightgbm available:', HAS_LGB)

# --- Paths -----------------------------------------------------------------
# In Colab, upload the 3 CSVs and point DATA_DIR at them, e.g.:
#   from google.colab import files; files.upload()
#   DATA_DIR = '/content'
# Locally we just point at the project ROUND_3 dir.
DATA_DIR = '/Users/emmett/Documents/Prosperity Trading/ROUND_3'
OUT_JSON = '/Users/emmett/Documents/Prosperity Trading/ml_fair_params.json'

# Hardcoded baselines from trader_round3.py — used as the constant baseline.
HARDCODED_FAIR = {
    'HYDROGEL_PACK': 9989.4,
    'VELVETFRUIT_EXTRACT': 5255.4,
    'VEV_4000': 1255.4,
    'VEV_4500': 750.0,
    'VEV_5000': 255.0,
    'VEV_5100': 167.0,
    'VEV_5200': 95.0,
    'VEV_5300': 46.5,
    'VEV_5400': 16.0,
    'VEV_5500': 6.6,
}


## 2. Data load

In [ ]:
frames = []
for d in (0, 1, 2):
    p = os.path.join(DATA_DIR, f'prices_round_3_day_{d}.csv')
    df = pd.read_csv(p, sep=';')
    df['day'] = d  # ensure day column is consistent even if csv has its own
    frames.append(df)
raw = pd.concat(frames, ignore_index=True)

num_cols = [c for c in raw.columns if c not in ('product',)]
for c in num_cols:
    raw[c] = pd.to_numeric(raw[c], errors='coerce')
raw = raw.sort_values(['product', 'day', 'timestamp']).reset_index(drop=True)
print(raw['product'].unique())
print(raw.shape)
raw.head()

## 3. Feature engineering

Per (product, day) compute orderbook stats, lags, and rolling moments.
All rolling/lag operations are restricted within a single day so we don't
leak across the day boundary.

In [ ]:
def add_book_features(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy()
    bid_p = g[['bid_price_1','bid_price_2','bid_price_3']].fillna(0).values
    bid_v = g[['bid_volume_1','bid_volume_2','bid_volume_3']].fillna(0).values
    ask_p = g[['ask_price_1','ask_price_2','ask_price_3']].fillna(0).values
    ask_v = g[['ask_volume_1','ask_volume_2','ask_volume_3']].fillna(0).values

    bid_notional = (bid_p * bid_v).sum(axis=1)
    ask_notional = (ask_p * ask_v).sum(axis=1)
    bid_vol = bid_v.sum(axis=1)
    ask_vol = ask_v.sum(axis=1)
    tot_vol = bid_vol + ask_vol
    tot_vol_safe = np.where(tot_vol == 0, 1, tot_vol)

    g['vwap'] = (bid_notional + ask_notional) / tot_vol_safe
    g['imbalance'] = (bid_vol - ask_vol) / tot_vol_safe
    g['spread'] = g['ask_price_1'] - g['bid_price_1']
    g['mid'] = g['mid_price']
    return g

def add_temporal_features(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values('timestamp').copy()
    mid = g['mid']
    for lag in (1, 5, 20):
        g[f'mid_lag_{lag}'] = mid - mid.shift(lag)
    for w in (50, 200):
        g[f'mid_roll_mean_{w}'] = mid.rolling(w, min_periods=5).mean()
        g[f'mid_roll_std_{w}']  = mid.rolling(w, min_periods=5).std()
    g['day_mid_mean'] = mid.expanding(min_periods=1).mean()  # running, used per-day
    return g

feat_frames = []
for (prod, day), g in raw.groupby(['product','day']):
    g2 = add_book_features(g)
    g2 = add_temporal_features(g2)
    feat_frames.append(g2)
feat = pd.concat(feat_frames, ignore_index=True)
print(feat.shape, feat.columns.tolist())

## 4. Target: 50-tick-ahead mid

In [ ]:
HORIZON = 50  # ticks

def add_target(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values('timestamp').copy()
    g['mid_future'] = g['mid'].shift(-HORIZON)
    return g

feat = feat.groupby(['product','day'], group_keys=False).apply(add_target)
feat = feat.dropna(subset=['mid_future']).reset_index(drop=True)
print('rows after dropna on target:', len(feat))

## 5. Train / test

Train on days {0,1}, test on day 2.  One model per product.  We fit a
small Ridge (linear, easy to port) and additionally a tree model for
comparison / feature importance.

In [ ]:
FEATURES = [
    'mid','vwap','imbalance','spread',
    'mid_lag_1','mid_lag_5','mid_lag_20',
    'mid_roll_mean_50','mid_roll_std_50',
    'mid_roll_mean_200','mid_roll_std_200',
    'day_mid_mean',
]

def fit_one(df_prod: pd.DataFrame):
    train = df_prod[df_prod['day'].isin([0,1])].dropna(subset=FEATURES)
    test  = df_prod[df_prod['day'] == 2].dropna(subset=FEATURES)
    if len(train) < 200 or len(test) < 50:
        return None

    Xtr, ytr = train[FEATURES].values, train['mid_future'].values
    Xte, yte = test[FEATURES].values,  test['mid_future'].values

    # Ridge — what we will actually port to trader.
    ridge = Ridge(alpha=1.0)
    ridge.fit(Xtr, ytr)
    yhat_r = ridge.predict(Xte)

    # Tree — for feature importance & a sanity ceiling.
    if HAS_LGB:
        tree = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05,
                                 num_leaves=31, min_child_samples=50,
                                 verbose=-1)
    else:
        tree = GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                         learning_rate=0.05)
    tree.fit(Xtr, ytr)
    yhat_t = tree.predict(Xte)

    # Constant baseline = whatever the trader currently hardcodes,
    # falling back to train-mean if we don't have one for this product.
    const = HARDCODED_FAIR.get(df_prod.name, ytr.mean())
    yhat_c = np.full_like(yte, const)

    out = {
        'product': df_prod.name,
        'n_train': len(train), 'n_test': len(test),
        'r2_ridge':    r2_score(yte, yhat_r),
        'mae_ridge':   mean_absolute_error(yte, yhat_r),
        'r2_tree':     r2_score(yte, yhat_t),
        'mae_tree':    mean_absolute_error(yte, yhat_t),
        'r2_const':    r2_score(yte, yhat_c),
        'mae_const':   mean_absolute_error(yte, yhat_c),
        'ridge_intercept': float(ridge.intercept_),
        'ridge_coef':      {f: float(c) for f, c in zip(FEATURES, ridge.coef_)},
        'tree_importance': dict(zip(FEATURES, getattr(tree, 'feature_importances_', np.zeros(len(FEATURES))).tolist())),
        'const_baseline':  float(const),
    }
    return out

results = {}
for prod, g in feat.groupby('product'):
    g.name = prod  # so fit_one can read df_prod.name
    r = fit_one(g)
    if r is not None:
        results[prod] = r
        print(f"{prod:25s} R2 ridge={r['r2_ridge']:.4f} tree={r['r2_tree']:.4f} const={r['r2_const']:.4f}  MAE ridge={r['mae_ridge']:.3f}")
    else:
        print(f"{prod:25s} skipped (insufficient data)")

## 6. Export portable params

We ship the **Ridge** model — intercept + per-feature coefficients — as JSON.
Re-implementation in `trader_round3.py` is a dot product over the same
feature list. We also pickle the tree models for offline experimentation
but they are NOT used at submit time.

In [ ]:
export = {
    'horizon': HORIZON,
    'features': FEATURES,
    'models': {
        prod: {
            'intercept': r['ridge_intercept'],
            'coef': r['ridge_coef'],
            'const_baseline': r['const_baseline'],
            'r2_ridge': r['r2_ridge'],
            'r2_const': r['r2_const'],
        }
        for prod, r in results.items()
    },
}
with open(OUT_JSON, 'w') as f:
    json.dump(export, f, indent=2)
print('wrote', OUT_JSON)

def fair_predict(features_dict, product, params=export):
    """Pure-Python predictor — mirror of what trader_round3.py should do."""
    m = params['models'][product]
    y = m['intercept']
    for f in params['features']:
        y += m['coef'][f] * float(features_dict.get(f, 0.0))
    return y

# Sanity check round-trip on a sample row
any_prod = next(iter(results))
sample = feat[feat['product']==any_prod].dropna(subset=FEATURES).iloc[-1]
print('sample fair_predict(', any_prod, ') =',
      fair_predict({f: sample[f] for f in FEATURES}, any_prod),
      'vs mid_future', sample['mid_future'])

## 7. Summary

In [ ]:
print('='*78)
print(f'{"product":25s} {"R2 ridge":>10s} {"R2 const":>10s} {"MAE ridge":>10s} {"MAE const":>10s}  top features')
print('-'*78)
for prod, r in sorted(results.items()):
    imp = r['tree_importance']
    top = sorted(imp.items(), key=lambda kv: -kv[1])[:3]
    top_str = ', '.join(f'{k}({v:.0f})' for k, v in top)
    improvement = r['r2_ridge'] - r['r2_const']
    flag = '  BEATS' if improvement > 0 else '  LOSES'
    print(f"{prod:25s} {r['r2_ridge']:10.4f} {r['r2_const']:10.4f} {r['mae_ridge']:10.3f} {r['mae_const']:10.3f}  {top_str}{flag}")
print('='*78)
print('R2_const < 0 means hardcoded constant is worse than predicting the test mean —')
print('strong signal that the hardcoded fair is biased and the ML model is worth porting.')